In [ ]:
from helper import redirect_tf_cpp_logs, VAE, scattering

redirect_tf_cpp_logs.redirect()
import re
import tensorflow as tf
import numpy
from matplotlib import pyplot, colors
import os
import json
import h5py


def read_logfile(filename):
    pat = re.compile(r"Epoch (\d+) \| TrainLoss: ([\d.]+), MSE: ([\d.]+), KL: ([\d.]+) \| " r"ValLoss: ([\d.]+), ValMSE: ([\d.]+), ValKL: ([\d.]+) \| Beta: ([\d.e+-]+)")
    rows = [tuple(map(float, m.groups())) for m in map(pat.search, open(filename)) if m]
    epochs, tr_loss, tr_mse, tr_kl, va_loss, va_mse, va_kl, beta = zip(*rows)
    tr_frac = [100 * b * k / l for b, k, l in zip(beta, tr_kl, tr_loss)]
    va_frac = [100 * b * k / l for b, k, l in zip(beta, va_kl, va_loss)]
    return epochs, tr_mse, tr_frac, tr_kl, va_frac, va_mse, va_kl

In [ ]:
epochs, tr_mse, tr_frac, tr_kl, va_frac, va_mse, va_kl = read_logfile("data/1crn_41bb_1d61dv_5co.vae/log.txt")
fig, ax = pyplot.subplots(1, 3, figsize=(8, 3), layout="tight")
ax[0].plot(epochs, tr_mse, label="train")
ax[0].plot(epochs, va_mse, label="val")
ax[0].set_yscale("log")
ax[0].set_title("Loss")
ax[0].legend()
ax[1].plot(epochs, tr_kl, label="train")
ax[1].plot(epochs, va_kl, label="val")
ax[1].set_title("Raw KL")
ax[1].legend()
ax[2].plot(epochs, tr_frac, label="train")
ax[2].plot(epochs, va_frac, label="val")
ax[2].set_title("KL share of total loss (%)")
ax[2].legend()
pyplot.show()

In [ ]:
model_dir = "data/1crn_41bb_1d61dv_5co.vae"
data_dir = "data/1crn_41bb_1d61dv_5co.tfrecord"

vae = VAE.load_vae(model_dir)
test_ds = VAE.make_test_dataset(os.path.join(data_dir, "tfrecords", "test.tfrecords"), batchsize=32)

X, R = [], []
for x in test_ds:
    X.append(x.numpy())
    R.append(VAE.reconstruct(vae, x).numpy())
X = numpy.concatenate(X)
R = numpy.concatenate(R)

In [ ]:
sel = X > 1e-5
diff = X[sel] - R[sel]
fig, ax = pyplot.subplots()
ax.hist(diff.flatten(), bins=1000)
ax.set(yscale="log", ylabel="Counts", xlabel="Prediction error (e/Å$^{3}$)", title="Test dataset")
pyplot.show()
del sel, diff

In [ ]:
norm = json.load(open(os.path.join(data_dir, "meta", "meta.json")))["normalization"]
xmin, xmax = norm["min"], norm["max"]
with h5py.File("data/1crn_41bb_1d61dv_5co.h5", "r") as f:
    dv = float(f["dv"][()])

index = numpy.random.randint(len(X))
x_phys = numpy.squeeze(X[index]) * (xmax - xmin) + xmin
r_phys = numpy.clip(numpy.squeeze(R[index]) * (xmax - xmin) + xmin, 0, None)

proj_x = x_phys.sum(axis=0)
proj_r = r_phys.sum(axis=0)
proj_d = proj_r - proj_x
vmax = max(proj_x.max(), proj_r.max())  # same colour scale for both densities
dmax = numpy.abs(proj_d).max()

fig, ax = pyplot.subplots(1, 4, figsize=(12, 4), layout="tight")
im0 = ax[0].imshow(x_phys.sum(axis=0), vmin=0, vmax=vmax)
ax[0].set_title(f"original (test {index})")
im1 = ax[1].imshow(r_phys.sum(axis=0), vmin=0, vmax=vmax)
ax[1].set_title("reconstruction")
im2 = ax[2].imshow((r_phys - x_phys).sum(axis=0), cmap="RdBu_r", vmin=-dmax, vmax=dmax)
ax[2].set_title("difference")


for a, im, title in zip(ax, [im0, im1, im2], [f"original (test {index})", "reconstruction", "difference"]):
    a.set_title(title)
    a.axis("off")
    fig.colorbar(im, ax=a, orientation="horizontal", location="bottom", shrink=0.8, label="e/Å^{3}")

q, I_orig = scattering.scattering_curve_gpu(x_phys, dv, rho_bulk=0.0, qmax=2)
_, I_rec = scattering.scattering_curve_gpu(r_phys, dv, rho_bulk=0.0, qmax=2)
q, I_orig, I_rec = q.get(), I_orig.get(), I_rec.get()

# fig, ax = pyplot.subplots(1, 2, figsize=(11, 4), layout="tight")
ax[3].plot(q, I_orig, label="original")
ax[3].plot(q, I_rec, label="reconstruction")
ax[3].set(yscale="log", xlabel="q (1/Å)", ylabel="I(q)")
ax[3].legend()
pyplot.show()

In [ ]:
support = (X > 0.01).any(axis=0)
outside = ~support
frac_out = R[:, outside].sum(axis=1) / R.reshape(len(R), -1).sum(axis=1)
print(f"density outside the protein: median {numpy.median(frac_out):.1%}, max {frac_out.max():.1%}")

In [ ]:
from scipy.ndimage import distance_transform_edt

voxel_size = 0.5  # your dv in Å
near, far = [], []
for x, r in zip(X[..., 0], R[..., 0]):
    dist = distance_transform_edt(x < 0.01) * voxel_size  # distance (Å) to the nearest occupied voxel
    total = r.sum()
    near.append(r[(dist > 0) & (dist <= 1.5)].sum() / total)
    far.append(r[dist > 1.5].sum() / total)
print(f"within 1.5 Å of the protein: median {numpy.median(near):.1%}")
print(f"further away:                median {numpy.median(far):.1%}")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from helper import VAE

model_dir = "data/1crn_41bb_1d61dv_5co.vae"
data_dir = "data/1crn_41bb_1d61dv_5co.tfrecord"
split = "train"  # or "test"

vae = VAE.load_vae(model_dir)

In [ ]:
def parse(rec):
    x = tf.io.parse_single_example(rec, {"data": tf.io.FixedLenFeature(VAE.GRID_SIZE**3, tf.float32)})["data"]
    return tf.reshape(x, VAE.INPUT_SHAPE)


path = os.path.join(data_dir, "tfrecords", f"{split}.tfrecords")
dataset = tf.data.TFRecordDataset(path).map(parse).batch(32)

z = []
for x in dataset:
    mean, logvar = vae.encode(x)
    z.append(mean.numpy())
z = np.concatenate(z)

print(z.shape)  # (number of samples, latent size)

In [ ]:
stats = np.column_stack([z.mean(axis=0), z.std(axis=0), z.min(axis=0), z.max(axis=0)])
np.savetxt(os.path.join(model_dir, "latent_stats.txt"), stats, header="mean std min max", fmt="%.6f")
print(stats)

In [ ]:
pca = PCA(n_components=2)
z2 = pca.fit_transform(z)
print("explained variance:", pca.explained_variance_ratio_)

plt.figure(figsize=(8, 6))
plt.scatter(z2[:, 0], z2[:, 1], c=np.arange(len(z2)), s=4, alpha=0.6)
plt.colorbar(label="frame index (file order)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Latent PCA")
plt.savefig(os.path.join(model_dir, "latent_pca.png"), dpi=300)
plt.show()